In [23]:
from getpass import getpass
from sqlalchemy import create_engine, URL, text

mysql_user = input("MySQL username: ")  # Often root

mysql_password = getpass("MySQL password: ")



In [24]:
%pip install pymysql

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
mysql_user = "root" 
mysql_password = getpass("Enter your MySQL password: ")

connection_url = URL.create(
    drivername="mysql+pymysql",
    username=mysql_user,
    password=mysql_password,
    host="localhost",
    port=3306,
    database="ecommerce_bronze"



)

In [26]:
import pymysql
print("PyMySQL is ready")

PyMySQL is ready


In [27]:


engine = create_engine(connection_url, pool_pre_ping=True)

with engine.connect() as connection:
    result = connection.execute(
        text("SELECT DATABASE(), VERSION()")
    ).one()

    print("Connected database:", result[0])
    print("MySQL version:", result[1])

Connected database: ecommerce_bronze
MySQL version: 8.0.46


In [37]:
import pandas as pd

In [38]:
from pathlib import Path
import hashlib

file_path = Path("../data/raw/ecommerce_customer (1).csv")

df_raw = pd.read_csv(
    file_path,
    dtype=str,
    keep_default_na=False
)

# Create fingerprint from file contents
with file_path.open("rb") as source_file:
    file_hash = hashlib.file_digest(
        source_file,
        "sha256"
    ).hexdigest()

print("Source file:", file_path.name)
print("Shape:", df_raw.shape)
print("File fingerprint:", file_hash)

Source file: ecommerce_customer (1).csv
Shape: (3055, 32)
File fingerprint: 1fe49872918fd3992ac24e3ace1b68506e523fe237b50bc8927dde7a4c6e3354


In [39]:
from sqlalchemy import text

with engine.begin() as connection:
    connection.execute(text("""
        CREATE TABLE IF NOT EXISTS ingestion_log (
            batch_id CHAR(36) PRIMARY KEY,
            file_name VARCHAR(255) NOT NULL,
            file_hash CHAR(64) NOT NULL UNIQUE,
            source_rows INT NOT NULL,
            loaded_rows INT NOT NULL,
            loaded_at TIMESTAMP NOT NULL
                DEFAULT CURRENT_TIMESTAMP
        ) ENGINE=InnoDB
    """))

print("Ingestion log is ready.")

Ingestion log is ready.


In [40]:
import uuid
from datetime import datetime

batch_id = str(uuid.uuid4())
loaded_at = datetime.now()

df_bronze = df_raw.copy()

df_bronze["batch_id"] = batch_id
df_bronze["source_file"] = file_path.name
df_bronze["loaded_at"] = loaded_at

print("Batch ID:", batch_id)
print("Bronze shape:", df_bronze.shape)

Batch ID: ac77e185-8531-4c50-bfb1-abbe9841d658
Bronze shape: (3055, 35)


In [41]:
bronze_table = "bronze_ecommerce_customer"

df_bronze.to_sql(
    name=bronze_table,
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=500,
    method="multi"
)

print(f"{len(df_bronze)} rows loaded into {bronze_table}")

3055 rows loaded into bronze_ecommerce_customer


In [42]:
with engine.connect() as connection:
    loaded_rows = connection.execute(
        text(f"SELECT COUNT(*) FROM {bronze_table}")
    ).scalar_one()

print("Source rows:", len(df_raw))
print("Loaded rows:", loaded_rows)

Source rows: 3055
Loaded rows: 3055


In [43]:
with engine.connect() as connection:
    log_df = pd.read_sql(
        text("""
            SELECT *
            FROM ingestion_log
            ORDER BY loaded_at DESC
        """),
        connection
    )

display(log_df)

,batch_id,file_name,file_hash,source_rows,loaded_rows,loaded_at
0,787bf869-b310-440d-9a23-6e117947bd13,ecommerce_customer (1).csv,1fe49872918fd3992ac24e3ace1b68506e523fe237b50b...,3055,3055,2026-10-03 19:16:19


In [44]:
with engine.begin() as connection:

    existing_batch = connection.execute(
        text("""
            SELECT batch_id
            FROM ingestion_log
            WHERE file_hash = :file_hash
        """),
        {"file_hash": file_hash}
    ).scalar()

    if existing_batch:
        print("File already ingested.")
        print("Existing batch ID:", existing_batch)

    else:
        connection.execute(
            text("""
                INSERT INTO ingestion_log (
                    batch_id,
                    file_name,
                    file_hash,
                    source_rows,
                    loaded_rows
                )
                VALUES (
                    :batch_id,
                    :file_name,
                    :file_hash,
                    :source_rows,
                    :loaded_rows
                )
            """),
            {
                "batch_id": batch_id,
                "file_name": file_path.name,
                "file_hash": file_hash,
                "source_rows": len(df_raw),
                "loaded_rows": loaded_rows
            }
        )

        print("Ingestion log updated successfully.")

File already ingested.
Existing batch ID: 787bf869-b310-440d-9a23-6e117947bd13


## Silver Layer - Data Quality Profiling

The Bronze layer preserves the raw source data.  
In the Silver layer, SQL profiling is performed to identify duplicates,
missing values, inconsistent categorical labels, and invalid business values
before applying cleaning rules.

In [45]:
with engine.connect() as connection:

    profile_summary = pd.read_sql(
        text("""
            SELECT
                COUNT(*) AS total_rows,
                COUNT(DISTINCT Order_ID) AS unique_orders,
                COUNT(*) - COUNT(DISTINCT Order_ID) AS duplicate_order_rows
            FROM bronze_ecommerce_customer
        """),
        connection
    )

display(profile_summary)

,total_rows,unique_orders,duplicate_order_rows
0,3055,3000,55


In [46]:
with engine.connect() as connection:

    duplicate_orders = pd.read_sql(
        text("""
            SELECT
                Order_ID,
                COUNT(*) AS row_count
            FROM bronze_ecommerce_customer
            GROUP BY Order_ID
            HAVING COUNT(*) > 1
            ORDER BY row_count DESC, Order_ID
        """),
        connection
    )

display(duplicate_orders)

,Order_ID,row_count
0,ORD100045,2
1,ORD100053,2
2,ORD100252,2
3,ORD100260,2
4,ORD100290,2
5,ORD100292,2
6,ORD100297,2
7,ORD100341,2
8,ORD100409,2
9,ORD100427,2


In [47]:
with engine.connect() as connection:

    duplicate_details = pd.read_sql(
        text("""
            SELECT *
            FROM bronze_ecommerce_customer
            WHERE Order_ID IN (
                SELECT Order_ID
                FROM bronze_ecommerce_customer
                GROUP BY Order_ID
                HAVING COUNT(*) > 1
            )
            ORDER BY Order_ID
        """),
        connection
    )

display(duplicate_details)

,Order_ID,Customer_ID,Order_Date,Customer_Age,Customer_Segment,City,Region,Category,Product_Name,Sales_Channel,...,Customer_Satisfaction,Gross_Amount,Discount_Amount,Net_Sales,Estimated_Cost,Profit,Profit_Margin_Pct,batch_id,source_file,loaded_at
0,ORD100045,C1152,2025-01-25,34.0,Premium,Bengaluru,South,Home Appliances,Shoes,Online,...,3.2,2371.46,29.88,2341.58,1517.05,824.53,35.21,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
1,ORD100045,C1152,2025-01-25,34.0,Premium,Bengaluru,South,Home Appliances,Shoes,Online,...,3.2,2371.46,29.88,2341.58,1517.05,824.53,35.21,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
2,ORD100053,C1602,2025-05-13,30.0,Regular,Bengaluru,North,Furniture,Treadmill,Online,...,4.5,18455.28,3532.34,14922.94,8850.48,6072.46,40.69,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
3,ORD100053,C1602,2025-05-13,30.0,Regular,Bengaluru,North,Furniture,Treadmill,Online,...,4.5,18455.28,3532.34,14922.94,8850.48,6072.46,40.69,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
4,ORD100252,C1598,2025-06-30,43.0,Regular,Kolkata,West,Books,Shirt,Online,...,4.3,6712.56,36.25,6676.31,5004.13,1672.18,25.05,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,ORD102777,C1319,2025-07-10,,Premium,Coimbatore,West,Fashion,Shirt,Online,...,3.8,208.46,59.39,149.07,117.27,31.8,21.33,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
106,ORD102852,C1472,2025-05-23,34.0,New,Chennai,North,Fashion,Headphones,Mobile App,...,,270.0,39.96,230.04,168.85,61.19,26.6,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
107,ORD102852,C1472,2025-05-23,34.0,New,Chennai,North,Fashion,Headphones,Mobile App,...,,270.0,39.96,230.04,168.85,61.19,26.6,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
108,ORD102896,C1639,2025-07-16,31.0,Regular,Delhi,South,Home Appliances,Rice,Marketplace,...,2.6,6431.88,1698.02,4733.86,2610.57,2123.29,44.85,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50


In [48]:
business_cols = [
    col for col in duplicate_details.columns
    if col not in ["batch_id", "source_file", "loaded_at"]
]

exact_duplicate_count = duplicate_details.duplicated(
    subset=business_cols,
    keep=False
).sum()

print("Rows participating in exact duplicates:", exact_duplicate_count)

Rows participating in exact duplicates: 100


In [49]:
extra_exact_duplicates = duplicate_details.duplicated(
    subset=business_cols,
    keep="first"
).sum()

print("Extra exact duplicate rows:", extra_exact_duplicates)

Extra exact duplicate rows: 50


In [50]:
non_exact_duplicates = duplicate_details[
    ~duplicate_details.duplicated(
        subset=business_cols,
        keep=False
    )
].copy()

display(non_exact_duplicates)

,Order_ID,Customer_ID,Order_Date,Customer_Age,Customer_Segment,City,Region,Category,Product_Name,Sales_Channel,...,Customer_Satisfaction,Gross_Amount,Discount_Amount,Net_Sales,Estimated_Cost,Profit,Profit_Margin_Pct,batch_id,source_file,loaded_at
18,ORD100427,C1385,2025-11-12,37.0,Regular,Bengaluru,Central,Sports,Mixer,Store,...,4.3,2842.42,812.93,2029.49,1665.59,363.9,17.93,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
19,ORD100427,C1385,2025-11-12,37.0,Regular,Bengaluru,Central,Sports,Mixer,Store,...,4.3,2842.42,812.93,2029.49,1665.59,363.9,17.93,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
26,ORD100700,C1608,2025-02-04,28.0,Regular,Coimbatore,South,Electronics,Skincare,Mobile App,...,3.8,1331.8,271.55,1060.25,805.28,254.97,24.05,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
27,ORD100700,C1608,2025-02-04,28.0,Regular,Coimbatore,South,Electronics,Skincare,Mobile App,...,3.8,1331.8,271.55,1060.25,805.28,254.97,24.05,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
66,ORD101794,C1207,2025-04-12,31.0,Occasional,Kolkata,South,Books,Headphones,Marketplace,...,4.6,700.32,840.38,-140.06,-103.38,-36.68,26.19,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
67,ORD101794,C1207,2025-04-12,31.0,Occasional,Kolkata,South,Books,Headphones,Marketplace,...,4.6,700.32,840.38,-140.06,-103.38,-36.68,26.19,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
76,ORD102113,C1794,2025-11-30,50.0,Regular,Delhi,North,Home Appliances,Novel,Online,...,,791.04,,,,,,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
77,ORD102113,C1794,2025-11-30,50.0,Regular,Delhi,North,Home Appliances,Novel,Online,...,,791.04,,,,,,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
80,ORD102116,C1210,2025-03-15,35.0,Occasional,Jaipur,South,Beauty,Novel,Store,...,3.6,4819.15,786.49,4032.66,3098.7,933.96,23.16,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
81,ORD102116,C1210,2025-03-15,35.0,Occasional,Jaipur,South,Beauty,Novel,Store,...,3.6,4819.15,786.49,4032.66,3098.7,933.96,23.16,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50


In [51]:
non_exact_duplicates = non_exact_duplicates.sort_values("Order_ID")

display(non_exact_duplicates)

,Order_ID,Customer_ID,Order_Date,Customer_Age,Customer_Segment,City,Region,Category,Product_Name,Sales_Channel,...,Customer_Satisfaction,Gross_Amount,Discount_Amount,Net_Sales,Estimated_Cost,Profit,Profit_Margin_Pct,batch_id,source_file,loaded_at
18,ORD100427,C1385,2025-11-12,37.0,Regular,Bengaluru,Central,Sports,Mixer,Store,...,4.3,2842.42,812.93,2029.49,1665.59,363.9,17.93,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
19,ORD100427,C1385,2025-11-12,37.0,Regular,Bengaluru,Central,Sports,Mixer,Store,...,4.3,2842.42,812.93,2029.49,1665.59,363.9,17.93,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
26,ORD100700,C1608,2025-02-04,28.0,Regular,Coimbatore,South,Electronics,Skincare,Mobile App,...,3.8,1331.8,271.55,1060.25,805.28,254.97,24.05,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
27,ORD100700,C1608,2025-02-04,28.0,Regular,Coimbatore,South,Electronics,Skincare,Mobile App,...,3.8,1331.8,271.55,1060.25,805.28,254.97,24.05,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
66,ORD101794,C1207,2025-04-12,31.0,Occasional,Kolkata,South,Books,Headphones,Marketplace,...,4.6,700.32,840.38,-140.06,-103.38,-36.68,26.19,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
67,ORD101794,C1207,2025-04-12,31.0,Occasional,Kolkata,South,Books,Headphones,Marketplace,...,4.6,700.32,840.38,-140.06,-103.38,-36.68,26.19,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
76,ORD102113,C1794,2025-11-30,50.0,Regular,Delhi,North,Home Appliances,Novel,Online,...,,791.04,,,,,,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
77,ORD102113,C1794,2025-11-30,50.0,Regular,Delhi,North,Home Appliances,Novel,Online,...,,791.04,,,,,,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
80,ORD102116,C1210,2025-03-15,35.0,Occasional,Jaipur,South,Beauty,Novel,Store,...,3.6,4819.15,786.49,4032.66,3098.7,933.96,23.16,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50
81,ORD102116,C1210,2025-03-15,35.0,Occasional,Jaipur,South,Beauty,Novel,Store,...,3.6,4819.15,786.49,4032.66,3098.7,933.96,23.16,ac77e185-8531-4c50-bfb1-abbe9841d658,ecommerce_customer (1).csv,2026-10-03 19:30:50


In [52]:
for order_id, group in non_exact_duplicates.groupby("Order_ID"):
    print("\nOrder:", order_id)

    for col in business_cols:
        if group[col].nunique(dropna=False) > 1:
            print(
                col,
                "->",
                group[col].tolist()
            )


Order: ORD100427
Order_Status -> ['Pending', 'Delivered']

Order: ORD100700
Order_Status -> ['Delivered', 'Delayed']

Order: ORD101794
Order_Status -> ['Delivered', 'Pending']

Order: ORD102113
Order_Status -> ['Delivered', 'Returned']

Order: ORD102116
Order_Status -> ['Delayed', 'Delivered']


In [53]:
conflicting_ids = non_exact_duplicates["Order_ID"].unique().tolist()

df_raw_conflicts = df_raw[
    df_raw["Order_ID"].isin(conflicting_ids)
].copy()

display(
    df_raw_conflicts[
        ["Order_ID", "Order_Status"]
    ]
)

,Order_ID,Order_Status
6,ORD102116,Delivered
171,ORD102116,Delayed
901,ORD100427,Pending
1151,ORD102113,Returned
1287,ORD102113,Delivered
1668,ORD101794,Pending
2037,ORD100427,Delivered
2206,ORD100700,Delayed
2529,ORD100700,Delivered
2782,ORD101794,Delivered


In [54]:
df_silver = df_bronze.drop_duplicates(
    subset="Order_ID",
    keep="last"
).copy()

print("Bronze rows:", len(df_bronze))
print("Silver rows after duplicate resolution:", len(df_silver))

Bronze rows: 3055
Silver rows after duplicate resolution: 3000


In [55]:
missing_profile = pd.DataFrame({
    "blank_count": [
        df_silver[col].astype(str).str.strip().eq("").sum()
        for col in df_silver.columns
    ]
}, index=df_silver.columns)

missing_profile = missing_profile[
    missing_profile["blank_count"] > 0
].sort_values("blank_count", ascending=False)

display(missing_profile)

,blank_count
Profit_Margin_Pct,240
Estimated_Cost,239
Discount_Amount,239
Net_Sales,239
Profit,239
Customer_Rating,209
Customer_Satisfaction,150
Discount_Pct,150
Customer_Loyalty_Points,120
Customer_Age,120


In [56]:
financial_cols = [
    "Quantity",
    "Unit_Price",
    "Discount_Pct",
    "Gross_Amount",
    "Discount_Amount",
    "Net_Sales",
    "Estimated_Cost",
    "Profit",
    "Profit_Margin_Pct"
]

financial_missing = df_silver[
    df_silver[financial_cols]
    .apply(lambda col: col.astype(str).str.strip().eq(""))
    .any(axis=1)
][financial_cols]

display(financial_missing.head(20))

print("Rows with at least one missing financial value:",
      len(financial_missing))

,Quantity,Unit_Price,Discount_Pct,Gross_Amount,Discount_Amount,Net_Sales,Estimated_Cost,Profit,Profit_Margin_Pct
1,2,,28.01,,,,,,
23,1,5160.28,,5160.28,,,,,
29,3,,25.52,,,,,,
33,5,1602.68,,8013.4,,,,,
35,4,1450.1,,5800.4,,,,,
39,2,402.93,,805.86,,,,,
43,4,1250.58,,5002.32,,,,,
46,2,324.54,,649.08,,,,,
53,2,,22.36,,,,,,
73,6,,4.72,,,,,,


Rows with at least one missing financial value: 240


In [57]:
missing_flags = pd.DataFrame({
    col: df_silver[col].astype(str).str.strip().eq("")
    for col in financial_cols
})

display(missing_flags.sum())

Quantity               0
Unit_Price            90
Discount_Pct         150
Gross_Amount          90
Discount_Amount      239
Net_Sales            239
Estimated_Cost       239
Profit               239
Profit_Margin_Pct    240
dtype: int64

In [58]:
display(
    missing_flags.value_counts().head(15)
)

Quantity  Unit_Price  Discount_Pct  Gross_Amount  Discount_Amount  Net_Sales  Estimated_Cost  Profit  Profit_Margin_Pct
False     False       False         False         False            False      False           False   False                2760
                      True          False         True             True       True            True    True                  149
          True        False         True          True             True       True            True    True                   89
          False       False         False         False            False      False           False   True                    1
          True        True          True          True             True       True            True    True                    1
Name: count, dtype: int64

In [59]:
case_discount_missing = df_silver[
    df_silver["Discount_Pct"].astype(str).str.strip().eq("")
][
    [
        "Order_ID",
        "Quantity",
        "Unit_Price",
        "Discount_Pct",
        "Gross_Amount",
        "Discount_Amount",
        "Net_Sales",
        "Estimated_Cost",
        "Profit",
        "Profit_Margin_Pct"
    ]
]

display(case_discount_missing.head(20))

,Order_ID,Quantity,Unit_Price,Discount_Pct,Gross_Amount,Discount_Amount,Net_Sales,Estimated_Cost,Profit,Profit_Margin_Pct
23,ORD101810,1,5160.28,,5160.28,,,,,
33,ORD100424,5,1602.68,,8013.4,,,,,
35,ORD100823,4,1450.1,,5800.4,,,,,
39,ORD101124,2,402.93,,805.86,,,,,
43,ORD102595,4,1250.58,,5002.32,,,,,
46,ORD101424,2,324.54,,649.08,,,,,
83,ORD100528,2,249.58,,499.16,,,,,
110,ORD101568,4,1638.12,,6552.48,,,,,
116,ORD100440,4,1539.72,,6158.88,,,,,
121,ORD100502,2,47.36,,94.72,,,,,


In [60]:
print(
    case_discount_missing[
        ["Quantity", "Unit_Price"]
    ].replace("", pd.NA).isna().sum()
)

Quantity      0
Unit_Price    1
dtype: int64


In [61]:
check_149 = case_discount_missing[
    case_discount_missing["Unit_Price"].astype(str).str.strip().ne("")
].copy()

check_149["calc_gross"] = (
    pd.to_numeric(check_149["Quantity"]) *
    pd.to_numeric(check_149["Unit_Price"])
)

check_149["gross_matches"] = (
    check_149["calc_gross"].round(2) ==
    pd.to_numeric(check_149["Gross_Amount"]).round(2)
)

print(check_149["gross_matches"].value_counts())

gross_matches
True    149
Name: count, dtype: int64


In [62]:
exceptional_discount_row = case_discount_missing[
    case_discount_missing["Unit_Price"].astype(str).str.strip().eq("")
]

display(exceptional_discount_row)

,Order_ID,Quantity,Unit_Price,Discount_Pct,Gross_Amount,Discount_Amount,Net_Sales,Estimated_Cost,Profit,Profit_Margin_Pct
2676,ORD102626,4,,,,,,,,


In [63]:
case_unit_price_missing = df_silver[
    df_silver["Unit_Price"].astype(str).str.strip().eq("")
][
    [
        "Order_ID",
        "Quantity",
        "Unit_Price",
        "Discount_Pct",
        "Gross_Amount",
        "Discount_Amount",
        "Net_Sales",
        "Estimated_Cost",
        "Profit",
        "Profit_Margin_Pct"
    ]
]

display(case_unit_price_missing.head(20))

print("Rows with missing Unit_Price:", len(case_unit_price_missing))

,Order_ID,Quantity,Unit_Price,Discount_Pct,Gross_Amount,Discount_Amount,Net_Sales,Estimated_Cost,Profit,Profit_Margin_Pct
1,ORD101072,2,,28.01,,,,,,
29,ORD102765,3,,25.52,,,,,,
53,ORD100026,2,,22.36,,,,,,
73,ORD100306,6,,4.72,,,,,,
88,ORD102159,1,,12.83,,,,,,
147,ORD101118,3,,4.97,,,,,,
154,ORD102756,3,,9.38,,,,,,
238,ORD101674,3,,17.91,,,,,,
249,ORD100441,4,,7.81,,,,,,
250,ORD100606,3,,12.54,,,,,,


Rows with missing Unit_Price: 90


In [64]:
print(
    case_unit_price_missing[
        ["Gross_Amount", "Discount_Pct"]
    ].replace("", pd.NA).isna().sum()
)

Gross_Amount    90
Discount_Pct     1
dtype: int64


In [65]:
product_price_profile = (
    df_silver[
        df_silver["Unit_Price"].astype(str).str.strip().ne("")
    ]
    .assign(
        Unit_Price_num=lambda x: pd.to_numeric(x["Unit_Price"])
    )
    .groupby("Product_Name")["Unit_Price_num"]
    .agg(
        count="count",
        unique_prices="nunique",
        min_price="min",
        max_price="max",
        median_price="median"
    )
    .sort_values("unique_prices", ascending=False)
)

display(product_price_profile)

,count,unique_prices,min_price,max_price,median_price
Product_Name,,,,,
Novel,324,323,29.48,13880.78,976.78
Treadmill,309,309,35.19,61182.50,981.91
Rice,307,307,12.75,11960.65,923.81
Shirt,298,298,47.36,21359.15,908.84
Chair,294,294,44.25,12288.53,992.71
Shoes,295,293,91.49,15454.69,870.39
Skincare,286,286,66.05,19021.37,909.28
Mixer,278,278,30.71,12119.85,899.03
Headphones,265,265,101.04,36945.75,939.38


In [68]:
case_unit_price_missing = df_silver[
    df_silver["Unit_Price"].astype(str).str.strip().eq("")
][
    [
        "Order_ID",
        "Product_Name",
        "Quantity",
        "Unit_Price",
        "Discount_Pct",
        "Gross_Amount",
        "Discount_Amount",
        "Net_Sales",
        "Estimated_Cost",
        "Profit",
        "Profit_Margin_Pct"
    ]
].copy()

In [69]:
missing_price_products = case_unit_price_missing[
    ["Order_ID", "Product_Name", "Quantity", "Discount_Pct"]
].merge(
    product_price_profile.reset_index(),
    on="Product_Name",
    how="left"
)

display(missing_price_products.head(20))

,Order_ID,Product_Name,Quantity,Discount_Pct,count,unique_prices,min_price,max_price,median_price
0,ORD101072,Shoes,2,28.01,295,293,91.49,15454.69,870.39
1,ORD102765,Shoes,3,25.52,295,293,91.49,15454.69,870.39
2,ORD100026,Rice,2,22.36,307,307,12.75,11960.65,923.81
3,ORD100306,Laptop,6,4.72,254,254,79.73,16416.62,848.78
4,ORD102159,Laptop,1,12.83,254,254,79.73,16416.62,848.78
5,ORD101118,Chair,3,4.97,294,294,44.25,12288.53,992.71
6,ORD102756,Shirt,3,9.38,298,298,47.36,21359.15,908.84
7,ORD101674,Laptop,3,17.91,254,254,79.73,16416.62,848.78
8,ORD100441,Mixer,4,7.81,278,278,30.71,12119.85,899.03
9,ORD100606,Chair,3,12.54,294,294,44.25,12288.53,992.71


In [71]:
age_profile = (
    df_silver[
        df_silver["Customer_Age"].astype(str).str.strip().ne("")
    ]
    .groupby("Customer_ID")["Customer_Age"]
    .agg(
        known_count="count",
        unique_ages="nunique",
        recovered_age="first"
    )
)

display(age_profile.head())

,known_count,unique_ages,recovered_age
Customer_ID,,,
C1001,3,3,30.0
C1002,1,1,44.0
C1003,4,4,41.0
C1004,4,4,45.0
C1005,2,2,38.0


In [72]:
missing_age_rows = df_silver[
    df_silver["Customer_Age"].astype(str).str.strip().eq("")
][
    ["Order_ID", "Customer_ID", "Customer_Age"]
].merge(
    age_profile.reset_index(),
    on="Customer_ID",
    how="left"
)

display(missing_age_rows.head(20))

,Order_ID,Customer_ID,Customer_Age,known_count,unique_ages,recovered_age
0,ORD100995,C1410,,3.0,3.0,36.0
1,ORD100823,C1439,,5.0,5.0,35.0
2,ORD101626,C1308,,1.0,1.0,31.0
3,ORD100494,C1596,,4.0,4.0,33.0
4,ORD101269,C1036,,3.0,3.0,26.0
5,ORD101744,C1262,,2.0,2.0,58.0
6,ORD101066,C1276,,3.0,3.0,43.0
7,ORD100968,C1194,,4.0,4.0,19.0
8,ORD101550,C1696,,3.0,3.0,34.0
9,ORD102219,C1042,,4.0,4.0,41.0


In [73]:
print(
    missing_age_rows["unique_ages"].value_counts(dropna=False)
)

unique_ages
2.0    29
3.0    27
4.0    21
5.0    16
1.0    12
6.0     7
7.0     7
NaN     1
Name: count, dtype: int64


120 missing Customer_Age rows

12 rows → safely recover from same Customer_ID
107 rows → conflicting age history, leave missing
1 row → no age history available, leave missing

In [74]:
safe_age_map = (
    age_profile[age_profile["unique_ages"] == 1]
    ["recovered_age"]
)

missing_age_mask = (
    df_silver["Customer_Age"]
    .astype(str)
    .str.strip()
    .eq("")
)

df_silver.loc[
    missing_age_mask &
    df_silver["Customer_ID"].isin(safe_age_map.index),
    "Customer_Age"
] = df_silver.loc[
    missing_age_mask &
    df_silver["Customer_ID"].isin(safe_age_map.index),
    "Customer_ID"
].map(safe_age_map)

print(
    "Remaining missing Customer_Age:",
    df_silver["Customer_Age"].astype(str).str.strip().eq("").sum()
)

Remaining missing Customer_Age: 108


In [75]:
age_consistency = (
    df_silver[
        df_silver["Customer_Age"].astype(str).str.strip().ne("")
    ]
    .groupby("Customer_ID")["Customer_Age"]
    .nunique()
)

df_silver["Age_Data_Quality"] = "Consistent"

df_silver.loc[
    df_silver["Customer_ID"].map(age_consistency).fillna(0) > 1,
    "Age_Data_Quality"
] = "Inconsistent"

In [76]:
age_ranges = (
    df_silver[
        df_silver["Customer_Age"].astype(str).str.strip().ne("")
    ]
    .assign(
        Customer_Age_num=lambda x: pd.to_numeric(x["Customer_Age"])
    )
    .groupby("Customer_ID")["Customer_Age_num"]
    .agg(
        min_age="min",
        max_age="max",
        unique_ages="nunique"
    )
)

age_ranges["age_gap"] = (
    age_ranges["max_age"] - age_ranges["min_age"]
)

display(
    age_ranges[
        age_ranges["unique_ages"] > 1
    ].sort_values("age_gap", ascending=False).head(20)
)

,min_age,max_age,unique_ages,age_gap
Customer_ID,,,,
C1736,19.0,102.0,4,83.0
C1553,20.0,102.0,4,82.0
C1075,13.0,88.0,8,75.0
C1035,27.0,102.0,2,75.0
C1260,28.0,102.0,5,74.0
C1048,20.0,88.0,7,68.0
C1566,1.0,61.0,4,60.0
C1120,9.0,68.0,5,59.0
C1470,-13.0,45.0,3,58.0


In [77]:
age_num = pd.to_numeric(
    df_silver["Customer_Age"],
    errors="coerce"
)

print(age_num.describe())

count    2892.000000
mean       35.334716
std        11.348204
min       -13.000000
25%        28.000000
50%        35.000000
75%        42.250000
max       102.000000
Name: Customer_Age, dtype: float64


In [78]:
print("Age < 0:", (age_num < 0).sum())
print("Age > 100:", (age_num > 100).sum())

Age < 0: 2
Age > 100: 4


In [79]:
age_num = pd.to_numeric(
    df_silver["Customer_Age"],
    errors="coerce"
)

invalid_age_mask = (
    (age_num < 13) |
    (age_num > 100)
)

df_silver.loc[
    invalid_age_mask,
    "Customer_Age"
] = pd.NA

print("Invalid/suspicious ages converted to missing:",
      invalid_age_mask.sum())

Invalid/suspicious ages converted to missing: 57


In [80]:
numeric_cols = [
    "Customer_Age",
    "Quantity",
    "Unit_Price",
    "Discount_Pct",
    "Shipping_Days",
    "Customer_Rating",
    "Delivery_Rating",
    "Customer_Loyalty_Points",
    "Previous_Orders",
    "Website_Session_Minutes",
    "Items_Viewed",
    "Customer_Satisfaction",
    "Gross_Amount",
    "Discount_Amount",
    "Net_Sales",
    "Estimated_Cost",
    "Profit",
    "Profit_Margin_Pct"
]

numeric_audit = []

for col in numeric_cols:

    values = pd.to_numeric(
        df_silver[col],
        errors="coerce"
    )

    numeric_audit.append({
        "column": col,
        "missing": values.isna().sum(),
        "min": values.min(),
        "max": values.max(),
        "mean": values.mean(),
        "median": values.median()
    })

numeric_audit = pd.DataFrame(numeric_audit)

display(numeric_audit)

,column,missing,min,max,mean,median
0,Customer_Age,165,13.00,88.00,35.759083,35.000
1,Quantity,0,-2.00,40.00,3.255667,3.000
2,Unit_Price,90,12.75,61182.50,1580.983058,922.740
3,Discount_Pct,150,-10.00,120.00,15.373761,14.890
4,Shipping_Days,105,-3.00,35.00,4.557858,4.000
5,Customer_Rating,209,0.00,9.00,3.745575,3.800
6,Delivery_Rating,0,1.00,5.00,3.676533,3.700
7,Customer_Loyalty_Points,120,0.00,2970.00,870.030556,828.000
8,Previous_Orders,0,0.00,13.00,5.004667,5.000
9,Website_Session_Minutes,75,1.00,96.10,14.713880,12.700


In [81]:
quantity_num = pd.to_numeric(
    df_silver["Quantity"],
    errors="coerce"
)

invalid_quantity = df_silver[
    quantity_num <= 0
][
    [
        "Order_ID",
        "Product_Name",
        "Quantity",
        "Unit_Price",
        "Gross_Amount",
        "Net_Sales",
        "Profit"
    ]
]

display(invalid_quantity)

print("Invalid Quantity rows:", len(invalid_quantity))

,Order_ID,Product_Name,Quantity,Unit_Price,Gross_Amount,Net_Sales,Profit
115,ORD101217,Headphones,0,874.82,0.0,0.0,0.0
1165,ORD101132,Headphones,-2,871.48,-1742.96,-1662.26,-606.6
1690,ORD102321,Laptop,-2,1267.15,-2534.3,-1987.4,-248.79
2045,ORD102085,Skincare,-2,3810.43,-7620.86,-6840.48,-1394.0


Invalid Quantity rows: 4


In [83]:
invalid_quantity_details = df_silver.loc[
    quantity_num <= 0,
    [
        "Order_ID",
        "Quantity",
        "Unit_Price",
        "Discount_Pct",
        "Gross_Amount",
        "Discount_Amount",
        "Net_Sales",
        "Estimated_Cost",
        "Profit",
        "Order_Status",
        "Return_Status"
    ]
]

display(invalid_quantity_details)

,Order_ID,Quantity,Unit_Price,Discount_Pct,Gross_Amount,Discount_Amount,Net_Sales,Estimated_Cost,Profit,Order_Status,Return_Status
115,ORD101217,0,874.82,9.04,0.0,0.0,0.0,0.0,0.0,Cancelled,No
1165,ORD101132,-2,871.48,4.63,-1742.96,-80.7,-1662.26,-1055.66,-606.6,Pending,No
1690,ORD102321,-2,1267.15,21.58,-2534.3,-546.9,-1987.4,-1738.61,-248.79,Delivered,No
2045,ORD102085,-2,3810.43,10.24,-7620.86,-780.38,-6840.48,-5446.48,-1394.0,Delivered,Yes


In [84]:
negative_qty_mask = pd.to_numeric(
    df_silver["Quantity"],
    errors="coerce"
) < 0

affected_financial_cols = [
    "Quantity",
    "Gross_Amount",
    "Discount_Amount",
    "Net_Sales",
    "Estimated_Cost",
    "Profit",
    "Profit_Margin_Pct"
]

df_silver.loc[
    negative_qty_mask,
    affected_financial_cols
] = pd.NA

print(
    "Negative quantity rows cleaned:",
    negative_qty_mask.sum()
)

Negative quantity rows cleaned: 3


In [85]:
discount_num = pd.to_numeric(
    df_silver["Discount_Pct"],
    errors="coerce"
)

invalid_discount = df_silver[
    (discount_num < 0) | (discount_num > 100)
][
    [
        "Order_ID",
        "Quantity",
        "Unit_Price",
        "Discount_Pct",
        "Gross_Amount",
        "Discount_Amount",
        "Net_Sales",
        "Estimated_Cost",
        "Profit",
        "Profit_Margin_Pct"
    ]
]

display(invalid_discount)

print("Invalid discount rows:", len(invalid_discount))

,Order_ID,Quantity,Unit_Price,Discount_Pct,Gross_Amount,Discount_Amount,Net_Sales,Estimated_Cost,Profit,Profit_Margin_Pct
499,ORD102676,4,492.7,-10.0,1970.8,-197.08,2167.88,1681.25,486.63,22.45
670,ORD100611,5,1507.16,120.0,7535.8,9042.96,-1507.16,-854.28,-652.88,43.32
961,ORD100633,2,855.67,120.0,1711.34,2053.61,-342.27,-254.04,-88.23,25.78
2629,ORD102047,2,131.2,-10.0,262.4,-26.24,288.64,242.52,46.12,15.98
2782,ORD101794,2,350.16,120.0,700.32,840.38,-140.06,-103.38,-36.68,26.19
2821,ORD101892,5,435.45,120.0,2177.25,2612.7,-435.45,-260.43,-175.02,40.19


Invalid discount rows: 6


In [86]:
discount_num = pd.to_numeric(
    df_silver["Discount_Pct"],
    errors="coerce"
)

invalid_discount_mask = (
    (discount_num < 0) |
    (discount_num > 100)
)

discount_affected_cols = [
    "Discount_Pct",
    "Discount_Amount",
    "Net_Sales",
    "Profit",
    "Profit_Margin_Pct"
]

df_silver.loc[
    invalid_discount_mask,
    discount_affected_cols
] = pd.NA

print(
    "Invalid discount rows cleaned:",
    invalid_discount_mask.sum()
)

Invalid discount rows cleaned: 6


In [87]:
shipping_num = pd.to_numeric(
    df_silver["Shipping_Days"],
    errors="coerce"
)

invalid_shipping = df_silver[
    shipping_num < 0
][
    [
        "Order_ID",
        "Shipping_Days",
        "Order_Status",
        "Return_Status",
        "Delivery_Rating",
        "Customer_Satisfaction"
    ]
]

display(invalid_shipping)

print("Invalid Shipping_Days rows:", len(invalid_shipping))

,Order_ID,Shipping_Days,Order_Status,Return_Status,Delivery_Rating,Customer_Satisfaction
1441,ORD102989,-3.0,Pending,No,2.0,3.5
1573,ORD100375,-3.0,Delivered,No,5.0,3.0


Invalid Shipping_Days rows: 2


In [88]:
shipping_num = pd.to_numeric(
    df_silver["Shipping_Days"],
    errors="coerce"
)

invalid_shipping_mask = shipping_num < 0

df_silver.loc[
    invalid_shipping_mask,
    "Shipping_Days"
] = pd.NA

print(
    "Invalid Shipping_Days cleaned:",
    invalid_shipping_mask.sum()
)

Invalid Shipping_Days cleaned: 2


In [89]:
rating_num = pd.to_numeric(
    df_silver["Customer_Rating"],
    errors="coerce"
)

invalid_rating = df_silver[
    (rating_num < 1) | (rating_num > 5)
][
    [
        "Order_ID",
        "Customer_Rating",
        "Delivery_Rating",
        "Customer_Satisfaction",
        "Order_Status",
        "Return_Status"
    ]
]

display(invalid_rating)

print("Invalid Customer_Rating rows:", len(invalid_rating))

,Order_ID,Customer_Rating,Delivery_Rating,Customer_Satisfaction,Order_Status,Return_Status
983,ORD102322,0.0,3.6,4.2,Delivered,No
1012,ORD101500,6.0,3.9,3.6,Pending,No
1114,ORD102616,0.0,4.0,2.8,Delayed,No
1423,ORD100572,9.0,5.0,3.9,Pending,No
1442,ORD101138,6.0,4.4,,Delivered,No
1768,ORD101854,0.0,2.2,3.5,Delivered,No
2124,ORD102427,6.0,3.5,2.7,Delivered,No


Invalid Customer_Rating rows: 7


In [90]:
rating_num = pd.to_numeric(
    df_silver["Customer_Rating"],
    errors="coerce"
)

invalid_rating_mask = (
    (rating_num < 1) |
    (rating_num > 5)
)

df_silver.loc[
    invalid_rating_mask,
    "Customer_Rating"
] = pd.NA

print(
    "Invalid Customer_Rating cleaned:",
    invalid_rating_mask.sum()
)

Invalid Customer_Rating cleaned: 7


In [91]:
other_numeric_cols = [
    "Delivery_Rating",
    "Customer_Satisfaction",
    "Customer_Loyalty_Points",
    "Previous_Orders",
    "Website_Session_Minutes",
    "Items_Viewed"
]

for col in other_numeric_cols:
    values = pd.to_numeric(df_silver[col], errors="coerce")

    print(
        col,
        "missing =", values.isna().sum(),
        "min =", values.min(),
        "max =", values.max()
    )

Delivery_Rating missing = 0 min = 1.0 max = 5.0
Customer_Satisfaction missing = 150 min = 1.0 max = 5.0
Customer_Loyalty_Points missing = 120 min = 0.0 max = 2970.0
Previous_Orders missing = 0 min = 0 max = 13
Website_Session_Minutes missing = 75 min = 1.0 max = 96.1
Items_Viewed missing = 0 min = 1 max = 18


In [92]:
unit_price_num = pd.to_numeric(
    df_silver["Unit_Price"],
    errors="coerce"
)

invalid_unit_price = df_silver[
    unit_price_num <= 0
][
    [
        "Order_ID",
        "Product_Name",
        "Quantity",
        "Unit_Price",
        "Gross_Amount",
        "Discount_Pct",
        "Net_Sales",
        "Profit"
    ]
]

display(invalid_unit_price)

print("Invalid Unit_Price rows:", len(invalid_unit_price))

,Order_ID,Product_Name,Quantity,Unit_Price,Gross_Amount,Discount_Pct,Net_Sales,Profit


Invalid Unit_Price rows: 0


In [93]:
estimated_cost_num = pd.to_numeric(
    df_silver["Estimated_Cost"],
    errors="coerce"
)

invalid_estimated_cost = df_silver[
    estimated_cost_num < 0
][
    [
        "Order_ID",
        "Quantity",
        "Unit_Price",
        "Discount_Pct",
        "Gross_Amount",
        "Net_Sales",
        "Estimated_Cost",
        "Profit",
        "Profit_Margin_Pct"
    ]
]

display(invalid_estimated_cost)

print(
    "Invalid Estimated_Cost rows:",
    len(invalid_estimated_cost)
)

,Order_ID,Quantity,Unit_Price,Discount_Pct,Gross_Amount,Net_Sales,Estimated_Cost,Profit,Profit_Margin_Pct
670,ORD100611,5,1507.16,NaN,7535.8,NaN,-854.28,NaN,NaN
961,ORD100633,2,855.67,NaN,1711.34,NaN,-254.04,NaN,NaN
2782,ORD101794,2,350.16,NaN,700.32,NaN,-103.38,NaN,NaN
2821,ORD101892,5,435.45,NaN,2177.25,NaN,-260.43,NaN,NaN


Invalid Estimated_Cost rows: 4


In [94]:
estimated_cost_num = pd.to_numeric(
    df_silver["Estimated_Cost"],
    errors="coerce"
)

invalid_estimated_cost_mask = estimated_cost_num < 0

df_silver.loc[
    invalid_estimated_cost_mask,
    "Estimated_Cost"
] = pd.NA

print(
    "Invalid Estimated_Cost cleaned:",
    invalid_estimated_cost_mask.sum()
)

Invalid Estimated_Cost cleaned: 4


In [95]:
qty_num = pd.to_numeric(df_silver["Quantity"], errors="coerce")
unit_num = pd.to_numeric(df_silver["Unit_Price"], errors="coerce")
gross_num = pd.to_numeric(df_silver["Gross_Amount"], errors="coerce")

valid_gross_inputs = (
    qty_num.notna() &
    unit_num.notna()
)

calc_gross = qty_num * unit_num

gross_mismatch_mask = (
    valid_gross_inputs &
    gross_num.notna() &
    (calc_gross.round(2) != gross_num.round(2))
)

print("Gross_Amount mismatches:", gross_mismatch_mask.sum())

Gross_Amount mismatches: 0


In [96]:
recoverable_gross_mask = (
    valid_gross_inputs &
    gross_num.isna()
)

print(
    "Recoverable missing Gross_Amount rows:",
    recoverable_gross_mask.sum()
)

Recoverable missing Gross_Amount rows: 0


In [97]:
discount_num = pd.to_numeric(
    df_silver["Discount_Pct"],
    errors="coerce"
)

discount_amount_num = pd.to_numeric(
    df_silver["Discount_Amount"],
    errors="coerce"
)

valid_discount_inputs = (
    gross_num.notna() &
    discount_num.notna()
)

calc_discount_amount = (
    gross_num * discount_num / 100
)

discount_amount_mismatch_mask = (
    valid_discount_inputs &
    discount_amount_num.notna() &
    (
        calc_discount_amount.round(2)
        != discount_amount_num.round(2)
    )
)

print(
    "Discount_Amount mismatches:",
    discount_amount_mismatch_mask.sum()
)

Discount_Amount mismatches: 0


In [98]:
recoverable_discount_amount_mask = (
    valid_discount_inputs &
    discount_amount_num.isna()
)

print(
    "Recoverable missing Discount_Amount rows:",
    recoverable_discount_amount_mask.sum()
)

Recoverable missing Discount_Amount rows: 0


In [99]:
net_sales_num = pd.to_numeric(
    df_silver["Net_Sales"],
    errors="coerce"
)

valid_net_sales_inputs = (
    gross_num.notna() &
    discount_amount_num.notna()
)

calc_net_sales = (
    gross_num - discount_amount_num
)

net_sales_mismatch_mask = (
    valid_net_sales_inputs &
    net_sales_num.notna() &
    (
        calc_net_sales.round(2)
        != net_sales_num.round(2)
    )
)

print(
    "Net_Sales mismatches:",
    net_sales_mismatch_mask.sum()
)

Net_Sales mismatches: 0


In [100]:
recoverable_net_sales_mask = (
    valid_net_sales_inputs &
    net_sales_num.isna()
)

print(
    "Recoverable missing Net_Sales rows:",
    recoverable_net_sales_mask.sum()
)

Recoverable missing Net_Sales rows: 0


In [101]:
profit_num = pd.to_numeric(
    df_silver["Profit"],
    errors="coerce"
)

estimated_cost_num = pd.to_numeric(
    df_silver["Estimated_Cost"],
    errors="coerce"
)

valid_profit_inputs = (
    net_sales_num.notna() &
    estimated_cost_num.notna()
)

calc_profit = (
    net_sales_num - estimated_cost_num
)

profit_mismatch_mask = (
    valid_profit_inputs &
    profit_num.notna() &
    (
        calc_profit.round(2)
        != profit_num.round(2)
    )
)

print(
    "Profit mismatches:",
    profit_mismatch_mask.sum()
)

Profit mismatches: 0


In [102]:
recoverable_profit_mask = (
    valid_profit_inputs &
    profit_num.isna()
)

print(
    "Recoverable missing Profit rows:",
    recoverable_profit_mask.sum()
)

Recoverable missing Profit rows: 0


In [103]:
margin_num = pd.to_numeric(
    df_silver["Profit_Margin_Pct"],
    errors="coerce"
)

valid_margin_inputs = (
    profit_num.notna() &
    net_sales_num.notna() &
    (net_sales_num != 0)
)

calc_margin = (
    profit_num / net_sales_num
) * 100

margin_mismatch_mask = (
    valid_margin_inputs &
    margin_num.notna() &
    (
        calc_margin.round(2)
        != margin_num.round(2)
    )
)

print(
    "Profit_Margin_Pct mismatches:",
    margin_mismatch_mask.sum()
)

Profit_Margin_Pct mismatches: 0


In [104]:
recoverable_margin_mask = (
    valid_margin_inputs &
    margin_num.isna()
)

print(
    "Recoverable missing Profit_Margin_Pct rows:",
    recoverable_margin_mask.sum()
)

Recoverable missing Profit_Margin_Pct rows: 0


In [105]:
final_numeric_audit = []

for col in numeric_cols:
    values = pd.to_numeric(
        df_silver[col],
        errors="coerce"
    )

    final_numeric_audit.append({
        "column": col,
        "missing": values.isna().sum(),
        "min": values.min(),
        "max": values.max()
    })

final_numeric_audit = pd.DataFrame(final_numeric_audit)

display(final_numeric_audit)

,column,missing,min,max
0,Customer_Age,165,13.00,88.00
1,Quantity,3,0.00,40.00
2,Unit_Price,90,12.75,61182.50
3,Discount_Pct,156,0.00,75.00
4,Shipping_Days,107,1.00,35.00
5,Customer_Rating,216,1.00,5.00
6,Delivery_Rating,0,1.00,5.00
7,Customer_Loyalty_Points,120,0.00,2970.00
8,Previous_Orders,0,0.00,13.00
9,Website_Session_Minutes,75,1.00,96.10


In [106]:
final_numeric_audit = []

for col in numeric_cols:
    values = pd.to_numeric(
        df_silver[col],
        errors="coerce"
    )

    final_numeric_audit.append({
        "column": col,
        "missing": values.isna().sum(),
        "min": values.min(),
        "max": values.max()
    })

final_numeric_audit = pd.DataFrame(final_numeric_audit)

display(final_numeric_audit)

,column,missing,min,max
0,Customer_Age,165,13.00,88.00
1,Quantity,3,0.00,40.00
2,Unit_Price,90,12.75,61182.50
3,Discount_Pct,156,0.00,75.00
4,Shipping_Days,107,1.00,35.00
5,Customer_Rating,216,1.00,5.00
6,Delivery_Rating,0,1.00,5.00
7,Customer_Loyalty_Points,120,0.00,2970.00
8,Previous_Orders,0,0.00,13.00
9,Website_Session_Minutes,75,1.00,96.10


In [107]:
df_silver["Financial_Data_Quality"] = "Complete"

financial_required = [
    "Quantity",
    "Unit_Price",
    "Discount_Pct",
    "Gross_Amount",
    "Discount_Amount",
    "Net_Sales",
    "Estimated_Cost",
    "Profit",
    "Profit_Margin_Pct"
]

financial_missing_mask = df_silver[
    financial_required
].apply(
    lambda col: pd.to_numeric(col, errors="coerce").isna()
).any(axis=1)

df_silver.loc[
    financial_missing_mask,
    "Financial_Data_Quality"
] = "Incomplete"

In [108]:
df_silver["Customer_Experience_Data_Quality"] = "Complete"

experience_cols = [
    "Customer_Rating",
    "Delivery_Rating",
    "Customer_Satisfaction"
]

experience_missing_mask = df_silver[
    experience_cols
].apply(
    lambda col: pd.to_numeric(col, errors="coerce").isna()
).any(axis=1)

df_silver.loc[
    experience_missing_mask,
    "Customer_Experience_Data_Quality"
] = "Incomplete"

In [109]:
df_silver["Record_Data_Quality"] = "Complete"

df_silver.loc[
    (df_silver["Financial_Data_Quality"] == "Incomplete") |
    (df_silver["Customer_Experience_Data_Quality"] == "Incomplete"),
    "Record_Data_Quality"
] = "Partial"

In [110]:
age_num = pd.to_numeric(
    df_silver["Customer_Age"],
    errors="coerce"
)

age_consistency = (
    df_silver[
        age_num.notna()
    ]
    .groupby("Customer_ID")["Customer_Age"]
    .nunique()
)

df_silver["Age_Data_Quality"] = "Consistent"

df_silver.loc[
    df_silver["Customer_Age"].isna(),
    "Age_Data_Quality"
] = "Missing"

df_silver.loc[
    df_silver["Customer_ID"].map(age_consistency).fillna(0) > 1,
    "Age_Data_Quality"
] = "Inconsistent"

In [111]:
print("Age Data Quality")
print(df_silver["Age_Data_Quality"].value_counts(dropna=False))

print("\nFinancial Data Quality")
print(df_silver["Financial_Data_Quality"].value_counts(dropna=False))

print("\nCustomer Experience Data Quality")
print(df_silver["Customer_Experience_Data_Quality"].value_counts(dropna=False))

print("\nOverall Record Data Quality")
print(df_silver["Record_Data_Quality"].value_counts(dropna=False))

Age Data Quality
Age_Data_Quality
Inconsistent    2889
Consistent       104
Missing            7
Name: count, dtype: int64

Financial Data Quality
Financial_Data_Quality
Complete      2751
Incomplete     249
Name: count, dtype: int64

Customer Experience Data Quality
Customer_Experience_Data_Quality
Complete      2645
Incomplete     355
Name: count, dtype: int64

Overall Record Data Quality
Record_Data_Quality
Complete    2429
Partial      571
Name: count, dtype: int64


In [112]:
checks = {
    "Negative Quantity": (
        pd.to_numeric(df_silver["Quantity"], errors="coerce") < 0
    ).sum(),

    "Invalid Discount": (
        (pd.to_numeric(df_silver["Discount_Pct"], errors="coerce") < 0) |
        (pd.to_numeric(df_silver["Discount_Pct"], errors="coerce") > 100)
    ).sum(),

    "Negative Shipping Days": (
        pd.to_numeric(df_silver["Shipping_Days"], errors="coerce") < 0
    ).sum(),

    "Invalid Customer Rating": (
        (pd.to_numeric(df_silver["Customer_Rating"], errors="coerce") < 1) |
        (pd.to_numeric(df_silver["Customer_Rating"], errors="coerce") > 5)
    ).sum(),

    "Negative Estimated Cost": (
        pd.to_numeric(df_silver["Estimated_Cost"], errors="coerce") < 0
    ).sum()
}

display(pd.DataFrame(
    checks.items(),
    columns=["Validation_Check", "Remaining_Invalid_Rows"]
))

,Validation_Check,Remaining_Invalid_Rows
0,Negative Quantity,0
1,Invalid Discount,0
2,Negative Shipping Days,0
3,Invalid Customer Rating,0
4,Negative Estimated Cost,0


## Numerical Data Quality - Summary

The numerical fields were profiled and validated before creating the Silver layer.

### Key issues identified
- Duplicate orders were resolved before numerical cleaning.
- Invalid customer ages and inconsistent age histories were identified.
- Negative quantities were treated as invalid, while zero quantity was retained for valid cancelled orders.
- Invalid discount percentages outside the 0–100% range were removed.
- Negative shipping durations were marked as missing.
- Customer ratings outside the valid 1–5 range were marked as missing.
- Negative estimated costs were treated as invalid.
- Missing numerical values were not blindly imputed using mean or median where no reliable business rule existed.

### Financial validation
Derived financial fields were validated using their business formulas:

- Gross Amount = Quantity × Unit Price
- Discount Amount = Gross Amount × Discount %
- Net Sales = Gross Amount − Discount Amount
- Profit = Net Sales − Estimated Cost
- Profit Margin % = (Profit / Net Sales) × 100

No formula mismatches remained where the required inputs were valid.

### Missing-value strategy
Missing values were retained when they could not be recovered reliably rather than introducing artificial values. Records were preserved because an order may still be useful for one type of analysis even when another field is incomplete.

Quality flags were therefore created for:
- Age Data Quality
- Financial Data Quality
- Customer Experience Data Quality
- Overall Record Data Quality

A final validation confirmed that no known invalid negative quantities, invalid discounts, negative shipping days, invalid customer ratings, or negative estimated costs remained.

The cleaned numerical data is now ready for categorical data-quality processing.

In [113]:
categorical_cols = [
    "Customer_Segment",
    "City",
    "Region",
    "Category",
    "Product_Name",
    "Sales_Channel",
    "Payment_Method",
    "Return_Status",
    "Order_Status",
    "Marketing_Source"
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(
        df_silver[col]
        .astype(str)
        .str.strip()
        .value_counts(dropna=False)
        .head(20)
    )


--- Customer_Segment ---
Customer_Segment
Regular       1459
Occasional     646
Premium        523
New            372
Name: count, dtype: int64

--- City ---
City
Chennai       305
Kochi         295
Coimbatore    292
Delhi         290
Kolkata       287
Hyderabad     284
Mumbai        284
Pune          276
Bengaluru     269
Jaipur        254
               74
bengaluru      15
chennai        13
pune           10
mumbai         10
hyderabad       9
kochi           8
kolkata         8
delhi           7
coimbatore      7
Name: count, dtype: int64

--- Region ---
Region
East       623
Central    612
West       597
North      587
South      581
Name: count, dtype: int64

--- Category ---
Category
Home Appliances    430
Sports             386
Grocery            376
Electronics        372
Furniture          363
Beauty             362
Fashion            357
Books              354
Name: count, dtype: int64

--- Product_Name ---
Product_Name
Novel         334
Treadmill     319
Rice          317


## Categorical Data Quality Processing

Categorical fields were profiled for inconsistent casing, leading/trailing spaces,
blank values, and inconsistent labels.

The main issues identified were:
- City names appearing with different capitalization.
- Payment methods appearing with different capitalization.
- Blank values in City and Payment_Method.

Standardization is applied while preserving genuinely missing values as NULL.

In [114]:
categorical_cols = [
    "Customer_Segment",
    "City",
    "Region",
    "Category",
    "Product_Name",
    "Sales_Channel",
    "Payment_Method",
    "Return_Status",
    "Order_Status",
    "Marketing_Source"
]

for col in categorical_cols:
    df_silver[col] = (
        df_silver[col]
        .astype("string")
        .str.strip()
        .replace("", pd.NA)
    )

In [115]:
df_silver["City"] = df_silver["City"].str.title()

df_silver["Payment_Method"] = (
    df_silver["Payment_Method"]
    .str.title()
)

In [116]:
print("City")
print(df_silver["City"].value_counts(dropna=False))

print("\nPayment Method")
print(df_silver["Payment_Method"].value_counts(dropna=False))

City
City
Chennai       318
Kochi         303
Coimbatore    299
Delhi         297
Kolkata       295
Mumbai        294
Hyderabad     293
Pune          286
Bengaluru     284
Jaipur        256
<NA>           74
Nan             1
Name: count, dtype: Int64

Payment Method
Payment_Method
Upi            533
Net Banking    505
Credit Card    486
Debit Card     469
Cash           466
Wallet         466
<NA>            75
Name: count, dtype: Int64


In [ ]:
# Convert literal missing-like strings to real missing values
for col in categorical_cols:
    df_silver[col] = df_silver[col].replace(
        ["nan", "Nan", "NAN", "None", "none"],
        pd.NA
    )

# Keep standard capitalization
df_silver["City"] = df_silver["City"].str.title()
df_silver["Payment_Method"] = df_silver["Payment_Method"].str.title()


df_silver["Payment_Method"] = (
    df_silver["Payment_Method"]
    .replace("Upi", "UPI")
)

In [119]:

print(df_silver["City"].value_counts(dropna=False))


print(df_silver["Payment_Method"].value_counts(dropna=False))

City
Chennai       318
Kochi         303
Coimbatore    299
Delhi         297
Kolkata       295
Mumbai        294
Hyderabad     293
Pune          286
Bengaluru     284
Jaipur        256
<NA>           75
Name: count, dtype: Int64
Payment_Method
UPI            533
Net Banking    505
Credit Card    486
Debit Card     469
Cash           466
Wallet         466
<NA>            75
Name: count, dtype: Int64


In [120]:
city_profile = (
    df_silver[
        df_silver["City"].notna()
    ]
    .groupby("Customer_ID")["City"]
    .agg(
        known_count="count",
        unique_cities="nunique",
        recovered_city="first"
    )
)

missing_city_rows = (
    df_silver[
        df_silver["City"].isna()
    ][
        ["Order_ID", "Customer_ID", "City"]
    ]
    .merge(
        city_profile.reset_index(),
        on="Customer_ID",
        how="left"
    )
)

display(missing_city_rows.head(20))

print(
    missing_city_rows["unique_cities"]
    .value_counts(dropna=False)
)

,Order_ID,Customer_ID,City,known_count,unique_cities,recovered_city
0,ORD101873,C1048,<NA>,6.0,4.0,Hyderabad
1,ORD100924,C1334,<NA>,2.0,2.0,Mumbai
2,ORD100591,C1090,<NA>,3.0,3.0,Kolkata
3,ORD101452,C1203,<NA>,3.0,2.0,Coimbatore
4,ORD101927,C1319,<NA>,3.0,3.0,Coimbatore
5,ORD102502,C1181,<NA>,5.0,3.0,Hyderabad
6,ORD100715,C1581,<NA>,3.0,3.0,Jaipur
7,ORD100228,C1303,<NA>,4.0,4.0,Hyderabad
8,ORD100022,C1105,<NA>,6.0,4.0,Hyderabad
9,ORD100721,C1512,<NA>,4.0,4.0,Kolkata


unique_cities
3.0    24
2.0    18
4.0    14
5.0    12
1.0     4
NaN     2
6.0     1
Name: count, dtype: int64


In [121]:
order_city_consistency = (
    df_silver[
        df_silver["City"].notna()
    ]
    .groupby("Order_ID")["City"]
    .nunique()
)

conflicting_order_cities = (
    order_city_consistency[
        order_city_consistency > 1
    ]
)

print(
    "Orders with multiple cities:",
    len(conflicting_order_cities)
)

display(conflicting_order_cities.head(20))

Orders with multiple cities: 0


Series([], Name: City, dtype: int64)

In [122]:
safe_city_map = (
    city_profile[
        city_profile["unique_cities"] == 1
    ]["recovered_city"]
)

missing_city_mask = df_silver["City"].isna()

safe_city_mask = (
    missing_city_mask &
    df_silver["Customer_ID"].isin(safe_city_map.index)
)

df_silver.loc[
    safe_city_mask,
    "City"
] = (
    df_silver.loc[
        safe_city_mask,
        "Customer_ID"
    ].map(safe_city_map)
)

print("City values recovered:", safe_city_mask.sum())
print("Remaining missing City:", df_silver["City"].isna().sum())

City values recovered: 4
Remaining missing City: 71


In [123]:
payment_profile = (
    df_silver[
        df_silver["Payment_Method"].notna()
    ]
    .groupby("Customer_ID")["Payment_Method"]
    .agg(
        known_count="count",
        unique_methods="nunique",
        recovered_method="first"
    )
)

missing_payment_rows = (
    df_silver[
        df_silver["Payment_Method"].isna()
    ][
        ["Order_ID", "Customer_ID", "Payment_Method"]
    ]
    .merge(
        payment_profile.reset_index(),
        on="Customer_ID",
        how="left"
    )
)

display(missing_payment_rows.head(20))

print(
    missing_payment_rows["unique_methods"]
    .value_counts(dropna=False)
)

,Order_ID,Customer_ID,Payment_Method,known_count,unique_methods,recovered_method
0,ORD102159,C1111,<NA>,2,2,Cash
1,ORD101799,C1313,<NA>,1,1,UPI
2,ORD100081,C1352,<NA>,4,4,Credit Card
3,ORD100440,C1664,<NA>,5,4,UPI
4,ORD102756,C1322,<NA>,7,4,Wallet
5,ORD101452,C1203,<NA>,3,3,Debit Card
6,ORD102110,C1229,<NA>,5,4,Debit Card
7,ORD100968,C1194,<NA>,5,4,Net Banking
8,ORD101823,C1727,<NA>,2,2,Credit Card
9,ORD100477,C1439,<NA>,5,4,Cash


unique_methods
3    23
4    18
2    13
1    13
5     7
6     1
Name: count, dtype: int64


In [124]:
safe_payment_map = (
    payment_profile[
        payment_profile["unique_methods"] == 1
    ]["recovered_method"]
)

missing_payment_mask = df_silver["Payment_Method"].isna()

safe_payment_mask = (
    missing_payment_mask &
    df_silver["Customer_ID"].isin(safe_payment_map.index)
)

df_silver.loc[
    safe_payment_mask,
    "Payment_Method"
] = (
    df_silver.loc[
        safe_payment_mask,
        "Customer_ID"
    ].map(safe_payment_map)
)

print("Payment methods recovered:", safe_payment_mask.sum())
print(
    "Remaining missing Payment_Method:",
    df_silver["Payment_Method"].isna().sum()
)

Payment methods recovered: 13
Remaining missing Payment_Method: 62


In [125]:
payment_counts = (
    df_silver[
        df_silver["Payment_Method"].notna()
    ]
    .groupby(
        ["Customer_ID", "Payment_Method"]
    )
    .size()
    .reset_index(name="method_count")
)

customer_totals = (
    payment_counts
    .groupby("Customer_ID")["method_count"]
    .sum()
    .rename("total_known_payments")
)

payment_preference = (
    payment_counts
    .sort_values(
        ["Customer_ID", "method_count"],
        ascending=[True, False]
    )
    .drop_duplicates("Customer_ID")
    .merge(
        customer_totals,
        on="Customer_ID"
    )
)

payment_preference["dominance_ratio"] = (
    payment_preference["method_count"] /
    payment_preference["total_known_payments"]
)

display(
    payment_preference[
        payment_preference["Customer_ID"].isin(
            df_silver.loc[
                df_silver["Payment_Method"].isna(),
                "Customer_ID"
            ]
        )
    ].head(20)
)

,Customer_ID,Payment_Method,method_count,total_known_payments,dominance_ratio
30,C1032,Debit Card,2,4,0.500000
43,C1045,UPI,2,4,0.500000
52,C1055,Cash,2,5,0.400000
63,C1067,Credit Card,1,2,0.500000
73,C1078,Cash,1,3,0.333333
78,C1083,Cash,2,4,0.500000
85,C1090,Credit Card,1,3,0.333333
101,C1107,UPI,2,5,0.400000
105,C1111,Cash,1,2,0.500000
123,C1129,Credit Card,2,6,0.333333


In [126]:
missing_customers = df_silver.loc[
    df_silver["Payment_Method"].isna(),
    "Customer_ID"
]

candidate_preferences = payment_preference[
    payment_preference["Customer_ID"].isin(missing_customers)
]

print(
    "100% consistent:",
    (candidate_preferences["dominance_ratio"] == 1).sum()
)

print(
    "At least 80% dominant:",
    (candidate_preferences["dominance_ratio"] >= 0.80).sum()
)

print(
    "At least 70% dominant:",
    (candidate_preferences["dominance_ratio"] >= 0.70).sum()
)

100% consistent: 0
At least 80% dominant: 0
At least 70% dominant: 3


# Missing Payment_Method values were recovered only when the customer
# had exactly one known payment method across historical orders.
# Remaining missing values were retained because payment method is
# order-specific and customer history showed multiple possible methods.

In [127]:
df_silver["Payment_Method_Data_Quality"] = "Observed"

df_silver.loc[
    df_silver["Payment_Method"].isna(),
    "Payment_Method_Data_Quality"
] = "Missing"

df_silver.loc[
    safe_payment_mask,
    "Payment_Method_Data_Quality"
] = "Recovered"

In [128]:
df_silver["City_Data_Quality"] = "Observed"

df_silver.loc[
    df_silver["City"].isna(),
    "City_Data_Quality"
] = "Missing"

df_silver.loc[
    safe_city_mask,
    "City_Data_Quality"
] = "Recovered"

In [129]:
silver_table = "silver_ecommerce_customer"

df_silver.to_sql(
    name=silver_table,
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=500,
    method="multi"
)

print(f"{len(df_silver)} rows loaded into {silver_table}")

3000 rows loaded into silver_ecommerce_customer


In [130]:
with engine.connect() as connection:
    silver_rows = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM silver_ecommerce_customer
        """)
    ).scalar_one()

print("Silver rows in MySQL:", silver_rows)

Silver rows in MySQL: 3000


In [148]:
df_silver["Return_Status_Data_Quality"] = "Consistent"

return_mismatch_mask = (
    (df_silver["Order_Status"] == "Returned") &
    (df_silver["Return_Status"] == "No")
)

df_silver.loc[
    return_mismatch_mask,
    "Return_Status_Data_Quality"
] = "Inconsistent"

print(
    df_silver["Return_Status_Data_Quality"]
    .value_counts()
)

Return_Status_Data_Quality
Consistent      2806
Inconsistent     194
Name: count, dtype: int64


In [149]:
df_silver.to_sql(
    name="silver_ecommerce_customer",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=500,
    method="multi"
)

print("Final Silver table refreshed.")

Final Silver table refreshed.


In [150]:
with engine.connect() as connection:
    silver_rows = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM silver_ecommerce_customer
        """)
    ).scalar_one()

print("Final Silver rows:", silver_rows)

Final Silver rows: 3000


In [135]:
gold_sql_path = Path("../sql/gold/01_profitability.sql")
gold_sql = gold_sql_path.read_text()

with engine.begin() as connection:
    for statement in gold_sql.split(";"):
        statement = statement.strip()

        if statement:
            connection.execute(text(statement))

print("Profitability Gold views created successfully.")

Profitability Gold views created successfully.


In [136]:
with engine.connect() as connection:

    category_gold = pd.read_sql(
        text("""
            SELECT *
            FROM gold_category_profitability
            ORDER BY total_profit DESC
        """),
        connection
    )

    product_gold = pd.read_sql(
        text("""
            SELECT *
            FROM gold_product_profitability
            ORDER BY total_profit DESC
        """),
        connection
    )

display(category_gold)
display(product_gold)

,Category,total_orders,total_net_sales,total_profit,profit_margin_pct,avg_discount_pct
0,Grocery,251,1228716.93,344752.98,28.06,14.95
1,Sports,262,1249252.62,338596.91,27.10,15.28
2,Fashion,243,1179504.39,330960.90,28.06,15.20
3,Furniture,237,1111013.45,312350.65,28.11,14.26
4,Books,243,1056020.84,303760.22,28.76,15.06
5,Electronics,248,995089.48,280755.45,28.21,15.42
6,Home Appliances,277,994985.40,274657.76,27.60,15.68
7,Beauty,234,968209.31,253927.44,26.23,15.72


,Product_Name,total_orders,total_net_sales,total_profit,profit_margin_pct,avg_discount_pct
0,Shirt,207,1156946.54,296865.68,25.66,14.93
1,Skincare,206,1014419.75,288627.30,28.45,15.69
2,Rice,219,961692.20,287301.01,29.87,15.41
3,Chair,203,967411.84,274863.14,28.41,14.78
4,Treadmill,209,990064.26,264141.73,26.68,14.81
5,Novel,217,852774.67,235939.28,27.67,15.25
6,Shoes,195,771357.65,213900.19,27.73,14.74
7,Headphones,186,755728.26,211480.40,27.98,15.39
8,Mixer,188,749991.81,210085.22,28.01,15.00
9,Laptop,165,562405.44,156558.36,27.84,16.19


In [133]:
product_category_check = (
    df_silver
    .groupby("Product_Name")["Category"]
    .nunique()
    .sort_values(ascending=False)
)

display(product_category_check)

print(
    "Products mapped to multiple categories:",
    (product_category_check > 1).sum()
)

Product_Name
Chair         8
Headphones    8
Laptop        8
Mixer         8
Novel         8
Rice          8
Shirt         8
Shoes         8
Skincare      8
Treadmill     8
Name: Category, dtype: int64

Products mapped to multiple categories: 10


In [137]:
discount_sql_path = Path(
    "../sql/gold/02_discount_profitability.sql"
)

discount_sql = discount_sql_path.read_text()

with engine.begin() as connection:
    connection.execute(text(discount_sql))

print("Discount profitability Gold view created successfully.")

Discount profitability Gold view created successfully.


In [138]:
with engine.connect() as connection:

    discount_gold = pd.read_sql(
        text("""
            SELECT *
            FROM gold_discount_profitability
            ORDER BY
                CASE discount_band
                    WHEN '0-5%' THEN 1
                    WHEN '5-10%' THEN 2
                    WHEN '10-15%' THEN 3
                    WHEN '15-20%' THEN 4
                    WHEN '20%+' THEN 5
                END
        """),
        connection
    )

display(discount_gold)

,discount_band,total_orders,avg_discount_pct,total_net_sales,total_profit,profit_margin_pct
0,0-5%,325,1.63,1686127.98,482155.85,28.60
1,5-10%,285,7.75,1421942.64,398934.39,28.06
2,10-15%,401,12.56,1792697.63,499067.79,27.84
3,15-20%,394,17.40,1662003.06,478390.86,28.78
4,20%+,590,26.61,2220021.11,581213.42,26.18


In [139]:
fulfilment_sql_path = Path(
    "../sql/gold/03_fulfilment_risk.sql"
)

fulfilment_sql = fulfilment_sql_path.read_text()

with engine.begin() as connection:
    connection.execute(text(fulfilment_sql))

print("Fulfilment risk Gold view created successfully.")

Fulfilment risk Gold view created successfully.


In [140]:
with engine.connect() as connection:

    fulfilment_gold = pd.read_sql(
        text("""
            SELECT *
            FROM gold_fulfilment_risk
            ORDER BY total_net_sales DESC
        """),
        connection
    )

display(fulfilment_gold)

,Order_Status,total_orders,total_net_sales,total_profit,net_sales_share_pct
0,Delivered,1995,8782792.42,2439762.31,71.84
1,Returned,210,1268250.29,403133.76,10.37
2,Pending,234,848103.31,237174.20,6.94
3,Cancelled,155,664674.10,209413.30,5.44
4,Delayed,157,662118.75,195970.94,5.42


In [141]:
risk_sql_path = Path(
    "../sql/gold/04_risk_concentration.sql"
)

risk_sql = risk_sql_path.read_text()

with engine.begin() as connection:
    connection.execute(text(risk_sql))

print("Risk concentration Gold view created successfully.")

Risk concentration Gold view created successfully.


In [142]:
with engine.connect() as connection:

    risk_gold = pd.read_sql(
        text("""
            SELECT *
            FROM gold_risk_concentration
            ORDER BY at_risk_net_sales DESC
            LIMIT 20
        """),
        connection
    )

display(risk_gold)

,Region,Category,Sales_Channel,Order_Status,total_orders,at_risk_net_sales
0,Central,Fashion,Marketplace,Returned,1,199210.22
1,South,Fashion,Store,Returned,4,75374.22
2,West,Sports,Online,Delayed,1,65405.62
3,West,Grocery,Store,Returned,3,65246.58
4,East,Home Appliances,Online,Returned,2,43632.56
5,North,Fashion,Online,Pending,3,43204.35
6,East,Home Appliances,Store,Delayed,1,40958.73
7,Central,Furniture,Online,Returned,5,40614.98
8,South,Home Appliances,Online,Cancelled,4,37222.58
9,West,Beauty,Store,Returned,2,31951.39


In [143]:
segment_sql_path = Path(
    "../sql/gold/05_customer_segment_performance.sql"
)

segment_sql = segment_sql_path.read_text()

with engine.begin() as connection:
    connection.execute(text(segment_sql))

print("Customer segment Gold view created successfully.")

Customer segment Gold view created successfully.


In [144]:
with engine.connect() as connection:

    segment_gold = pd.read_sql(
        text("""
            SELECT *
            FROM gold_customer_segment_performance
            ORDER BY total_profit DESC
        """),
        connection
    )

display(segment_gold)

,Customer_Segment,total_orders,avg_previous_orders,avg_loyalty_points,avg_customer_satisfaction,total_net_sales,total_profit,profit_margin_pct
0,Regular,991,4.94,851.93,3.52,4357569.55,1196006.30,27.45
1,Occasional,424,5.16,845.22,3.43,1883113.20,538255.60,28.58
2,Premium,337,4.95,834.40,3.55,1390512.56,384445.84,27.65
3,New,243,5.05,743.60,3.54,1151597.11,321054.57,27.88


In [145]:
fulfilment_exp_sql_path = Path(
    "../sql/gold/06_fulfilment_experience.sql"
)

fulfilment_exp_sql = fulfilment_exp_sql_path.read_text()

with engine.begin() as connection:
    connection.execute(text(fulfilment_exp_sql))

print("Fulfilment experience Gold view created successfully.")

Fulfilment experience Gold view created successfully.


In [146]:
with engine.connect() as connection:

    fulfilment_exp_gold = pd.read_sql(
        text("""
            SELECT *
            FROM gold_fulfilment_experience
            ORDER BY total_orders DESC
        """),
        connection
    )

display(fulfilment_exp_gold)

,Order_Status,total_orders,avg_shipping_days,avg_delivery_rating,avg_customer_satisfaction,returned_orders,return_rate_pct
0,Delivered,2167,4.38,3.67,3.50,286.0,13.20
1,Pending,258,4.56,3.70,3.54,33.0,12.79
2,Returned,227,4.42,3.72,3.43,33.0,14.54
3,Delayed,176,4.26,3.68,3.37,25.0,14.20
4,Cancelled,172,4.59,3.61,3.39,29.0,16.86


In [147]:
status_return_check = pd.crosstab(
    df_silver["Order_Status"],
    df_silver["Return_Status"],
    margins=True
)

display(status_return_check)

Return_Status,No,Yes,All
Order_Status,,,
Cancelled,143,29,172
Delayed,151,25,176
Delivered,1881,286,2167
Pending,225,33,258
Returned,194,33,227
All,2594,406,3000


In [152]:
gold_files = [
    "01_profitability.sql",
    "02_discount_profitability.sql",
    "03_fulfilment_risk.sql",
    "04_risk_concentration.sql",
    "05_customer_segment_performance.sql",
    "06_fulfilment_experience.sql"
]

for file_name in gold_files:

    sql_path = Path("../sql/gold") / file_name
    sql_code = sql_path.read_text()

    with engine.begin() as connection:
        for statement in sql_code.split(";"):
            statement = statement.strip()

            if statement:
                connection.execute(text(statement))

    print(f"{file_name} executed successfully.")

01_profitability.sql executed successfully.
02_discount_profitability.sql executed successfully.
03_fulfilment_risk.sql executed successfully.
04_risk_concentration.sql executed successfully.
05_customer_segment_performance.sql executed successfully.
06_fulfilment_experience.sql executed successfully.


## Gold Layer - Business Analytics Summary

The Gold layer was created in MySQL using SQL views built from the cleaned Silver dataset.

Only trusted Silver records and relevant quality rules were used for business analysis.

### Gold Views Created

1. **Category / Product Profitability**
   - Delivered orders only
   - Net Sales
   - Total Profit
   - Profit Margin %
   - Average Discount %

2. **Discount vs Profitability**
   - Orders grouped into discount bands
   - Used to identify whether higher discounting is associated with weaker margins

3. **Fulfilment Risk**
   - Net sales associated with Delivered, Pending, Delayed, Returned, and Cancelled orders
   - Helps quantify revenue exposure by order status

4. **Risk Concentration**
   - Identifies high-risk combinations by Region, Category, Sales Channel, and Order Status

5. **Customer Segment Performance**
   - Previous-order behaviour
   - Loyalty points
   - Customer satisfaction
   - Net sales
   - Profitability

6. **Fulfilment Experience**
   - Shipping days
   - Delivery rating
   - Customer satisfaction
   - Return flag rate by order status

### Important Data Quality Considerations

- Missing or invalid values were not blindly imputed.
- Financial analysis uses records marked as financially complete.
- Product and Category are treated as separate analytical dimensions because the source data does not provide a consistent product-to-category hierarchy.
- `Order_Status` and `Return_Status` are retained as separate signals because they are not fully consistent with each other.
- Recovered City and Payment Method values are explicitly flagged in Silver.

The Gold layer is now ready to be consumed by Power BI for business reporting and dashboard creation.